In [1]:
!git clone https://gitlab.com/mjslee0921/flowpacker

Cloning into 'flowpacker'...
remote: Enumerating objects: 214, done.
remote: Counting objects: 100% (71/71), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 214 (delta 34), reused 67 (delta 32), pack-reused 143 (from 1)
Receiving objects: 100% (214/214), 1.04 MiB | 20.08 MiB/s, done.
Resolving deltas: 100% (51/51), done.
Filtering content: 100% (3/3), 585.68 MiB | 145.25 MiB/s, done.


In [2]:
%cd flowpacker

/kaggle/working/flowpacker


In [3]:
!sed -i 's/numpy==1.22.4/numpy>=1.22.4/g' requirements.txt

In [4]:
!pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 399.2/399.2 kB 11.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 99.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 450.7/450.7 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 93.9 MB/s eta 0:00:00
  Created wheel for torch_cluster: filename=torch_cluster-1.6.3-cp312-cp312-linux_x86_64.whl size=2205953 sha256=b

In [5]:
!pip install pdbfixer
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 665.9/665.9 kB 11.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.3/14.3 MB 92.3 MB/s eta 0:00:00
  Created wheel for pdbfixer: filename=pdbfixer-1.12.0-py3-none-any.whl size=681683 sha256=c15f44eb113b1321a3c6e63ed7da4a35f52e8ed7a2a6718c1c7ab29f8c050061
  Stored in directory: /root/.cache/pip/wheels/79/ec/63/2ad240b7fac835f490bd1ecb748da87c99ae65618770c73a98
Successfully built pdbfixer


In [6]:
!mkdir /content/flowpacker/pdbs
!cp -r /kaggle/input/datasets/calistu/frames/* /content/flowpacker/pdbs 
!ls /content/flowpacker/pdbs

mkdir: cannot create directory ‘/content/flowpacker/pdbs’: No such file or directory
cp: target '/content/flowpacker/pdbs' is not a directory
ls: cannot access '/content/flowpacker/pdbs': No such file or directory


In [7]:
import os
import glob
from pdbfixer import PDBFixer
from openmm.app import PDBFile

# 1. Definir diretórios de forma estrita
input_dir = '/content/flowpacker/pdbs' # Sua pasta atual com os arquivos do BioEmu
output_dir = '/content/flowpacker/pdbs_fixed' # Nova pasta apenas para os dados limpos
os.makedirs(output_dir, exist_ok=True)

# 2. Mapear todos os arquivos .pdb originais
pdb_files = sorted(glob.glob(os.path.join(input_dir, '*.pdb')))

print(f"Encontrados {len(pdb_files)} arquivos. Iniciando reconstrução da topologia full-atom...")

# 3. Loop de processamento em lote
for filepath in pdb_files:
    filename = os.path.basename(filepath)
    out_filepath = os.path.join(output_dir, filename)

    try:
        # Carrega, encontra os buracos na matriz e preenche com as coordenadas padronizadas
        fixer = PDBFixer(filename=filepath)
        fixer.findMissingResidues()
        fixer.findMissingAtoms()
        fixer.addMissingAtoms()

        # Salva o arquivo impecável no novo diretório
        with open(out_filepath, 'w') as f:
            PDBFile.writeFile(fixer.topology, fixer.positions, f)

    except Exception as e:
        print(f"[ERRO FATAL] Falha na matriz do arquivo {filename}: {e}")

print(f"\nProcessamento concluído. Seus {len(pdb_files)} frames estão na pasta '{output_dir}/'.")

Encontrados 0 arquivos. Iniciando reconstrução da topologia full-atom...

Processamento concluído. Seus 0 frames estão na pasta '/content/flowpacker/pdbs_fixed/'.


In [8]:
!sed -i 's/torch.load(self.config.ckpt)/torch.load(self.config.ckpt, weights_only=False)/g' sampler_pdb.py


In [ ]:
!sed -i 's|/mnt/d/ab_validation_pdbs|pdbs_fixed|g' sampler_pdb.py


In [9]:
!python sampler_pdb.py base 1000_samples

/kaggle/working/flowpacker/models/equiformer_v2/layer_norm.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(enabled=False)
/kaggle/working/flowpacker/models/equiformer_v2/layer_norm.py:152: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(enabled=False)
/kaggle/working/flowpacker/models/equiformer_v2/layer_norm.py:227: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(enabled=False)
/kaggle/working/flowpacker/models/equiformer_v2/layer_norm.py:310: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(enabled=False)
Traceback (most recent call last):
  File "/kaggle/working/flowpacker/sam